# SpotFake combined


## 0. Зависимости

In [ ]:
%%capture
!pip install torch transformers scikit-learn pandas numpy matplotlib seaborn pyarrow catboost

## 1. Импорты и конфигурация

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, classification_report
)
from catboost import CatBoostClassifier
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
BERT_MODEL    = 'DeepPavlov/rubert-base-cased'
MAX_LEN       = 128
BERT_BATCH    = 64
IMG_DIM       = 512
EMB_DIM       = 128
DROPOUT       = 0.3
BATCH_SIZE    = 256
EPOCHS        = 30
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
WARMUP_EPOCHS = 3

Device: cuda


## 2. Загрузка данных

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
df = pd.read_csv('/content/drive/MyDrive/ozon_train.csv', encoding='utf-8')
print(f'Данные: {df.shape}')
print(f'Доля контрафакта: {df["resolution"].mean():.4f}')
clip_emb = pd.read_parquet('/content/drive/MyDrive/clip_embeddings.parquet')
clip_emb['vector'] = clip_emb['embedding'].apply(lambda x: np.array(x, dtype=np.float32))
print(f'CLIP embeddings: {clip_emb.shape}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Данные: (197198, 45)
Доля контрафакта: 0.0662
CLIP embeddings: (187604, 3)


## 3. Seller-based разбиение (идентично v2)

In [ ]:
seller_class_counts = df.groupby('SellerID')['resolution'].nunique()
multi_class_sellers = seller_class_counts[seller_class_counts > 1].index
np.random.seed(SEED)
test_sellers  = np.random.choice(multi_class_sellers,
size=int(0.3 * len(multi_class_sellers)),
replace=False)
train_sellers = np.setdiff1d(df['SellerID'].unique(), test_sellers)
train_df = df[df['SellerID'].isin(train_sellers)].copy().reset_index(drop=True)
test_df  = df[df['SellerID'].isin(test_sellers)].copy().reset_index(drop=True)
print(f'Train: {train_df.shape}, контрафакт: {train_df["resolution"].mean():.4f}')
print(f'Test:  {test_df.shape},  контрафакт: {test_df["resolution"].mean():.4f}')

Train: (177380, 45), контрафакт: 0.0588
Test:  (19818, 45),  контрафакт: 0.1321


## 4. Подготовка признаков

### 4.1 Seller LOO target encoding (из v2)

In [ ]:
global_mean = train_df['resolution'].mean()
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
seller_loo = np.full(len(train_df), global_mean)
for _, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
    fold_rate = train_df.iloc[tr_idx].groupby('SellerID')['resolution'].mean()
    seller_loo[val_idx] = train_df.iloc[val_idx]['SellerID'].map(fold_rate).fillna(global_mean).values
train_df['seller_problem_rate'] = seller_loo
seller_global = train_df.groupby('SellerID')['resolution'].mean().reset_index()
seller_global.columns = ['SellerID', 'seller_problem_rate']
test_df = test_df.merge(seller_global, on='SellerID', how='left')
test_df['seller_problem_rate'] = test_df['seller_problem_rate'].fillna(global_mean)
print(f'seller_problem_rate корреляция: {train_df["seller_problem_rate"].corr(train_df["resolution"]):.4f}')

seller_problem_rate корреляция: 0.7107


### 4.2 Brand LOO target encoding (новое)

EDA: `brand_name` — значимый сигнал контрафакта. Применяем LOO аналогично seller.

In [ ]:
brand_loo = np.full(len(train_df), global_mean)
for _, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
    fold_rate = train_df.iloc[tr_idx].groupby('brand_name')['resolution'].mean()
    brand_loo[val_idx] = train_df.iloc[val_idx]['brand_name'].map(fold_rate).fillna(global_mean).values
train_df['brand_problem_rate'] = brand_loo
brand_global = train_df.groupby('brand_name')['resolution'].mean().reset_index()
brand_global.columns = ['brand_name', 'brand_problem_rate']
test_df = test_df.merge(brand_global, on='brand_name', how='left')
test_df['brand_problem_rate'] = test_df['brand_problem_rate'].fillna(global_mean)
print(f'brand_problem_rate корреляция: {train_df["brand_problem_rate"].corr(train_df["resolution"]):.4f}')

brand_problem_rate корреляция: 0.4216


### 4.3 Feature engineering

In [ ]:
def engineer_tabular(df_):
    out = df_.copy()
    for col in ['brand_name', 'description', 'GmvTotal7', 'GmvTotal30', 'GmvTotal90', 'rating_5_count']:
        if col in out.columns:
            out[f'is_null_{col}'] = out[col].isnull().astype(np.float32)
    for w in [7, 30, 90]:
        sales = out.get(f'item_count_sales{w}', pd.Series(0, index=out.index))
        ret   = out.get(f'item_count_returns{w}', pd.Series(0, index=out.index))
        fake  = out.get(f'item_count_fake_returns{w}', pd.Series(0, index=out.index))
        out[f'return_rate{w}']         = ret  / (sales + 1)
        out[f'fake_return_rate{w}']    = fake / (sales + 1)
        out[f'fake_to_all_returns{w}'] = fake / (ret + 1)
    for col in ['item_time_alive', 'seller_time_alive', 'PriceDiscounted']:
        if col in out.columns:
            out[f'log_{col}'] = np.log1p(out[col].fillna(0))
    out['log_age_interaction'] = out['log_item_time_alive'] * out['log_seller_time_alive']
    out['description_length']     = out['description'].fillna('').str.len().astype(np.float32)
    out['name_length']            = out['name_rus'].fillna('').str.len().astype(np.float32)
    out['brand_length']           = out['brand_name'].fillna('').str.len().astype(np.float32)
    out['has_description']        = (out['description'].notna() & (out['description'].str.len() > 10)).astype(np.float32)
    out['has_brand']              = out['brand_name'].notna().astype(np.float32)
    out['log_description_length'] = np.log1p(out['description_length'])
    out['log_name_length']        = np.log1p(out['name_length'])
    if 'PriceDiscounted' in out.columns:
        seller_med = out.groupby('SellerID')['PriceDiscounted'].transform('median')
        out['price_vs_seller_median'] = out['PriceDiscounted'] / (seller_med + 1)
    if 'GmvTotal90' in out.columns and 'GmvTotal7' in out.columns:
        out['gmv_7_to_90'] = out['GmvTotal7'].fillna(0) / (out['GmvTotal90'].fillna(0) + 1)
    return out
train_df = engineer_tabular(train_df)
test_df  = engineer_tabular(test_df)
print(f'Итого признаков: {train_df.shape[1]}')

Итого признаков: 75


### 4.4 ruBERT с mean pooling (вместо CLS)

In [ ]:
%%capture
tokenizer  = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def make_text(df_):
    brand = df_['brand_name'].fillna('без бренда')
    return df_['name_rus'].fillna('') + ' ' + df_['description'].fillna('') + ' ' + brand
def get_bert_embeddings(texts, batch_size=BERT_BATCH, max_length=MAX_LEN):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i:i+batch_size].tolist()
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            out = bert_model(**encoded)
        mask = encoded['attention_mask'].unsqueeze(-1).float()
        emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        all_embs.append(emb.cpu().numpy())
        if i % 10000 == 0:
            print(f'  {i}/{len(texts)}')
    return np.vstack(all_embs).astype(np.float32)
train_df['text'] = make_text(train_df)
test_df['text']  = make_text(test_df)
X_train_text = get_bert_embeddings(train_df['text'].values)
X_test_text  = get_bert_embeddings(test_df['text'].values)
TEXT_DIM = X_train_text.shape[1]
print(f'Эмбеддинги: {X_train_text.shape}')

Кодируем train...
  0/177380
  40000/177380
  80000/177380
  120000/177380
  160000/177380
Кодируем test...
  0/19818
Эмбеддинги: (177380, 768)


### 4.5 CLIP + финальная сборка признаков

In [ ]:
train_clip = train_df[['ItemID', 'resolution']].merge(
    clip_emb[['ItemID', 'vector']], on='ItemID', how='inner')
test_clip  = test_df[['ItemID', 'resolution']].merge(
    clip_emb[['ItemID', 'vector']], on='ItemID', how='inner')
train_mask = train_df['ItemID'].isin(train_clip['ItemID']).values
test_mask  = test_df['ItemID'].isin(test_clip['ItemID']).values
y_tr = train_clip['resolution'].values.astype(np.float32)
y_te = test_clip['resolution'].values.astype(np.float32)
X_tr_text = X_train_text[train_mask]
X_te_text = X_test_text[test_mask]
X_tr_img  = np.vstack(train_clip['vector'].values)
X_te_img  = np.vstack(test_clip['vector'].values)
drop_cols = {'id', 'ItemID', 'SellerID', 'name_rus', 'description',
             'brand_name', 'CommercialTypeName4', 'text', 'resolution'}
num_cols  = [
    c for c in train_df.columns
    if c not in drop_cols and train_df[c].dtype in ['float64', 'int64', 'float32']
]
X_tr_tab = train_df[train_mask][num_cols].fillna(-1).values.astype(np.float32)
X_te_tab = test_df[test_mask][num_cols].fillna(-1).values.astype(np.float32)
TAB_DIM  = X_tr_tab.shape[1]
print(f'Табличных признаков: {TAB_DIM}')
print(f'Train: {len(y_tr)}, контрафакт: {y_tr.mean():.4f}')
scalers = {}
for name, tr, te in [('text', X_tr_text, X_te_text),
                      ('img',  X_tr_img,  X_te_img),
                      ('tab',  X_tr_tab,  X_te_tab)]:
    sc = StandardScaler()
    tr[:] = sc.fit_transform(tr)
    te[:] = sc.transform(te)
    scalers[name] = sc

Табличных признаков: 67
Train: 168619, контрафакт: 0.0586


## 5. Dataset и DataLoader

In [ ]:
class OzonDataset(Dataset):
    def __init__(self, text, img, tab, labels):
        self.text   = torch.tensor(text,   dtype=torch.float32)
        self.img    = torch.tensor(img,    dtype=torch.float32)
        self.tab    = torch.tensor(tab,    dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
    def __len__(self):         return len(self.labels)
    def __getitem__(self, i):  return self.text[i], self.img[i], self.tab[i], self.labels[i]
train_dataset = OzonDataset(X_tr_text, X_tr_img, X_tr_tab, y_tr)
test_dataset  = OzonDataset(X_te_text, X_te_img, X_te_tab, y_te)
class_counts   = np.bincount(y_tr.astype(int))
class_weights  = 1.0 / class_counts
sample_weights = class_weights[y_tr.astype(int)]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(sample_weights), replacement=True
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,   num_workers=2)
print(f'Train batches: {len(train_loader)}')

Train batches: 659


## 6. Архитектура

```
Текст (TEXT_DIM) ──► TextEncoder(512→128) ──┐
Картинка (512)   ──► ImgEncoder(256→128)  ──┼──► Concat(384) ──► Classifier ──► sigmoid
Таблица (~66)    ──► TabEncoder(256→128)  ──┘
```
Cross-modal attention убран — нестабилен без предобучения на этих данных.

In [ ]:
class ModalityEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=128, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.LayerNorm(output_dim),
        )
    def forward(self, x): return self.net(x)
class OzonV3(nn.Module):
    def __init__(self, text_dim, img_dim, tab_dim, emb_dim=EMB_DIM, dropout=DROPOUT):
        super().__init__()
        self.text_encoder  = ModalityEncoder(text_dim, 512, emb_dim, dropout)
        self.image_encoder = ModalityEncoder(img_dim,  256, emb_dim, dropout)
        self.tab_encoder   = ModalityEncoder(tab_dim,  256, emb_dim, dropout)

        fused_dim = emb_dim * 3  # 384
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, 1)
        )

    def forward(self, text, img, tab):
        fused = torch.cat([
            self.text_encoder(text),
            self.image_encoder(img),
            self.tab_encoder(tab),
        ], dim=1)
        return self.classifier(fused).squeeze(-1)
model = OzonV3(text_dim=TEXT_DIM, img_dim=IMG_DIM, tab_dim=TAB_DIM).to(device)
print(f'Параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Параметров: 792,321


## 7. Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.80):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits, targets):
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs   = torch.sigmoid(logits)
        p_t     = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()
criterion = FocalLoss(gamma=2.0, alpha=0.80)
print('Focal Loss: gamma=2.0, alpha=0.80 (идентично v2)')

Focal Loss: gamma=2.0, alpha=0.80 (идентично v2)


## 8. Обучение с warmup (3 эп.) + CosineAnnealing

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for text, img, tab, y in loader:
            logits = model(text.to(device), img.to(device), tab.to(device))
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y.numpy())
    y_pred = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)
    return roc_auc_score(y_true, y_pred), average_precision_score(y_true, y_pred)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
warmup   = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine   = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-5)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
best_pr_auc = 0.0
best_state  = None
history = {'train_loss': [], 'val_roc': [], 'val_pr': []}
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for text, img, tab, y in train_loader:
        optimizer.zero_grad()
        logits = model(text.to(device), img.to(device), tab.to(device))
        loss   = criterion(logits, y.to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    val_roc, val_pr = evaluate(model, test_loader, device)
    history['train_loss'].append(avg_loss)
    history['val_roc'].append(val_roc)
    history['val_pr'].append(val_pr)
    if val_pr > best_pr_auc:
        best_pr_auc = val_pr
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d} | Loss: {avg_loss:.4f} | ROC-AUC: {val_roc:.4f} | PR-AUC: {val_pr:.4f} | LR: {lr_now:.6f}')
print(f'\nЛучший PR-AUC нейросети: {best_pr_auc:.4f}')

Epoch   1 | Loss: 0.0262 | ROC-AUC: 0.9025 | PR-AUC: 0.5981 | LR: 0.000120
Epoch   5 | Loss: 0.0070 | ROC-AUC: 0.9094 | PR-AUC: 0.6418 | LR: 0.000296
Epoch  10 | Loss: 0.0044 | ROC-AUC: 0.9107 | PR-AUC: 0.6628 | LR: 0.000255
Epoch  15 | Loss: 0.0030 | ROC-AUC: 0.9099 | PR-AUC: 0.6636 | LR: 0.000180
Epoch  20 | Loss: 0.0022 | ROC-AUC: 0.9114 | PR-AUC: 0.6636 | LR: 0.000098
Epoch  25 | Loss: 0.0016 | ROC-AUC: 0.9082 | PR-AUC: 0.6767 | LR: 0.000034
Epoch  30 | Loss: 0.0015 | ROC-AUC: 0.9097 | PR-AUC: 0.6813 | LR: 0.000010

Лучший PR-AUC нейросети: 0.6925


## 9. CatBoost на тех же признаках

In [ ]:
X_tr_cb = train_df[train_mask][num_cols].fillna(-1)
X_te_cb = test_df[test_mask][num_cols].fillna(-1)
pos_weight = float((y_tr == 0).sum()) / float((y_tr == 1).sum())
cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=7,
    l2_leaf_reg=3,
    scale_pos_weight=pos_weight,
    eval_metric='PRAUC',
    early_stopping_rounds=100,
    random_seed=SEED,
    verbose=200,
    task_type='GPU' if torch.cuda.is_available() else 'CPU'
)
cb.fit(X_tr_cb, y_tr, eval_set=(X_te_cb, y_te))
cb_probs_te = cb.predict_proba(X_te_cb)[:, 1]
cb_roc = roc_auc_score(y_te, cb_probs_te)
cb_pr  = average_precision_score(y_te, cb_probs_te)
print(f'CatBoost: ROC-AUC={cb_roc:.4f}, PR-AUC={cb_pr:.4f}')

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.9450090	test: 0.8769907	best: 0.8769907 (0)	total: 344ms	remaining: 11m 28s
bestTest = 0.9188967426
bestIteration = 34
Shrink model to first 35 iterations.
CatBoost: ROC-AUC=0.8315, PR-AUC=0.4944


## 10. Ансамбль + калибровка порога

In [ ]:
model.load_state_dict(best_state)
model.eval()
nn_probs = []
with torch.no_grad():
    for text, img, tab, y in test_loader:
        logits = model(text.to(device), img.to(device), tab.to(device))
        nn_probs.append(torch.sigmoid(logits).cpu().numpy())
nn_probs = np.concatenate(nn_probs)
y_true   = y_te
best_pr, best_alpha = 0.0, 0.6
for alpha in np.arange(0.2, 1.0, 0.05):
    probs = alpha * nn_probs + (1 - alpha) * cb_probs_te
    pr    = average_precision_score(y_true, probs)
    if pr > best_pr:
        best_pr    = pr
        best_alpha = alpha
y_pred = best_alpha * nn_probs + (1 - best_alpha) * cb_probs_te
print(f'Лучший alpha (нейросеть): {best_alpha:.2f}, PR-AUC: {best_pr:.4f}')

Подбор веса ансамбля...
Лучший alpha (нейросеть): 0.95, PR-AUC: 0.6939


In [ ]:
precision_arr, recall_arr, thresholds = precision_recall_curve(y_true, y_pred)
mask = precision_arr[:-1] >= 0.9
if mask.any():
    best_idx       = np.argmax(recall_arr[:-1][mask])
    best_threshold = thresholds[mask][best_idx]
    recall_at_90   = recall_arr[:-1][mask][best_idx]
    prec_at_thresh = precision_arr[:-1][mask][best_idx]
else:
    best_threshold, recall_at_90, prec_at_thresh = 0.5, 0.0, 0.0
print(f'Оптимальный порог: {best_threshold:.4f}')
print(f'При пороге: Precision={prec_at_thresh:.4f}, Recall={recall_at_90:.4f}')

Оптимальный порог: 0.9228
При пороге: Precision=0.9006, Recall=0.1265


## 11. Итоговая оценка

In [ ]:
roc = roc_auc_score(y_true, y_pred)
pr  = average_precision_score(y_true, y_pred)
print(f'ROC-AUC:                      {roc:.4f}')
print(f'PR-AUC (ансамбль):            {pr:.4f}')
print(f'  — только нейросеть:         {best_pr_auc:.4f}')
print(f'  — только CatBoost:          {cb_pr:.4f}')
print(f'Recall@P≥0.9 (порог={best_threshold:.3f}): {recall_at_90:.4f}')
print()
print(classification_report(
    y_true, (y_pred >= best_threshold).astype(int),
    target_names=['Легитимный', 'Контрафакт']
))

## 12. Сравнение всех версий

In [ ]:
results = pd.DataFrame([
    {'Модель': 'Baseline CatBoost',    'ROC-AUC': 0.9051, 'PR-AUC': 0.6231, 'Recall@P≥0.9': 0.0145},
    {'Модель': 'SpotFake-Ozon',        'ROC-AUC': 0.9134, 'PR-AUC': 0.6660, 'Recall@P≥0.9': 0.0000},
    {'Модель': 'SAFE-Ozon v1',         'ROC-AUC': 0.9131, 'PR-AUC': 0.6564, 'Recall@P≥0.9': 0.0012},
    {'Модель': 'Spotfake combined', 'ROC-AUC': roc,    'PR-AUC': pr,     'Recall@P≥0.9': recall_at_90},
]).set_index('Модель')
results['PR-AUC Δ vs SpotFake'] = (results['PR-AUC'] - 0.6660).round(4)
display(results.style
    .highlight_max(subset=['ROC-AUC', 'PR-AUC', 'Recall@P≥0.9'], color='lightgreen')
    .format('{:.4f}', na_rep='—'))

## 13. Ablation: вклад новых компонентов

In [ ]:
def eval_ablation(model, loader, device, zero_indices=None):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for text, img, tab, y in loader:
            tab = tab.to(device)
            if zero_indices:
                tab = tab.clone(); tab[:, zero_indices] = 0
            logits = model(text.to(device), img.to(device), tab)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y.numpy())
    y_p = np.concatenate(all_probs); y_l = np.concatenate(all_labels)
    return roc_auc_score(y_l, y_p), average_precision_score(y_l, y_p)
new_text_feats  = ['description_length','name_length','brand_length',
                   'has_description','has_brand','log_description_length','log_name_length']
text_feat_idx   = [i for i, c in enumerate(num_cols) if c in new_text_feats]
brand_rate_idx  = [i for i, c in enumerate(num_cols) if 'brand_problem' in c]
seller_rate_idx = [i for i, c in enumerate(num_cols) if 'seller_problem' in c]
ratio_idx       = [i for i, c in enumerate(num_cols) if 'rate' in c or 'fake_to_all' in c]
rows = []
for label, idx in [
    ('Все признаки v3',           None),
    ('Без text-length признаков', text_feat_idx),
    ('Без brand_problem_rate',    brand_rate_idx),
    ('Без seller_problem_rate',   seller_rate_idx),
    ('Без ratio features',        ratio_idx),
]:
    r, p = eval_ablation(model, test_loader, device, idx)
    rows.append({'Конфигурация': label, 'ROC-AUC': r, 'PR-AUC': p})
    print(f'{label}: ROC-AUC={r:.4f}, PR-AUC={p:.4f}')
display(pd.DataFrame(rows).set_index('Конфигурация').style
    .highlight_max(subset=['ROC-AUC','PR-AUC'], color='lightgreen').format('{:.4f}'))

## 14. Feature importance CatBoost

In [ ]:
fi = pd.Series(cb.get_feature_importance(), index=num_cols).sort_values(ascending=False)

new_feats = ['description_length','name_length','brand_length','has_description','has_brand',
             'log_description_length','log_name_length','brand_problem_rate',
             'price_vs_seller_median','gmv_7_to_90']

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#2ecc71' if f in new_feats else 'steelblue' for f in fi.head(25).index]
fi.head(25).plot(kind='barh', ax=ax, color=colors)
ax.invert_yaxis()
ax.set_title('CatBoost feature importance (зелёный = новое в v3)')
ax.set_xlabel('Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('catboost_fi_v3.png', dpi=150, bbox_inches='tight')
plt.show()
